# Tuya Cloud Export
Exports all devices, scenes, automations, and related data from a Tuya developer account.

**Required env vars:**
- `TUYA_ACCESS_ID` — Cloud project Access ID (from Tuya IoT console)
- `TUYA_ACCESS_KEY` — Cloud project Access Secret
- `TUYA_API_ENDPOINT` — Regional endpoint (default: `https://openapi.tuyaeu.com`)
  - EU: `https://openapi.tuyaeu.com`
  - US: `https://openapi.tuyaus.com`
  - CN: `https://openapi.tuyacn.com`
  - IN: `https://openapi.tuyain.com`
- `TUYA_USER_UID` — (optional) User UID from the Tuya IoT console Device Management > Users
- `TUYA_HOME_IDS` — (optional) Comma-separated list of home IDs to query


In [ ]:
%pip install tuya-connector-python dotenv --quiet

In [ ]:
import os
import json
import time
import datetime
import csv
from pathlib import Path
from pprint import pprint
from tuya_connector import TuyaOpenAPI

from dotenv import load_dotenv
load_dotenv("../.env")  # or absolute path

# Credentials from env
ACCESS_ID       = os.environ["TUYA_ACCESS_ID"]
ACCESS_KEY      = os.environ["TUYA_ACCESS_KEY"]
API_ENDPOINT    = os.environ.get("TUYA_API_ENDPOINT", "https://openapi.tuyaeu.com")
USER_UID        = os.environ.get("TUYA_USER_UID", "")
HOME_IDS_ENV    = os.environ.get("TUYA_HOME_IDS", "")

print(f"Endpoint : {API_ENDPOINT}")
print(f"AccessID : {ACCESS_ID[:6]}...")
print(f"User UID : {USER_UID or '(not set)'}")
print(f"Home IDs : {HOME_IDS_ENV or '(not set)'}")

In [ ]:
# Connect
openapi = TuyaOpenAPI(API_ENDPOINT, ACCESS_ID, ACCESS_KEY)
resp = openapi.connect()
print("Connect response:", resp)

In [ ]:
USER_UID = resp['result']['uid']
print(f"UID: {USER_UID}")

## Helpers

In [ ]:
def ok(resp):
    """Return True if the API call succeeded."""
    return bool(resp and resp.get("success", False))


def result(resp):
    """Return the 'result' payload, or None on failure."""
    if ok(resp):
        return resp.get("result")
    return None


def extract_list(data, *keys):
    """Extract a list from a dict by trying multiple candidate keys."""
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for k in keys:
            if k in data:
                return data[k]
        # Try common names as fallback
        for k in ("list", "devices", "homes", "scenes", "automations",
                  "rules", "items", "records", "result"):
            if k in data:
                return data[k]
    return []


def get_paged(path, params=None, page_size=100):
    """Fetch all pages (page_no / page_size style)."""
    params = dict(params or {})
    params["page_size"] = page_size
    records = []
    page = 1
    while True:
        params["page_no"] = page
        resp = openapi.get(path, params)
        if not ok(resp):
            print(f"  [WARN] {path} page={page}: {resp.get('msg', resp)}")
            break
        data = resp["result"]
        batch = extract_list(data)
        records.extend(batch)
        total = data.get("total", len(batch)) if isinstance(data, dict) else len(batch)
        has_more = data.get("has_more", False) if isinstance(data, dict) else False
        if not has_more and len(records) >= total:
            break
        if len(batch) < page_size:
            break
        page += 1
    return records


def get_cursor_paged(path, params=None, page_size=50):
    """Cursor-style pagination via last_row_key (iot-01 endpoints)."""
    params = dict(params or {})
    params["page_size"] = page_size
    records = []
    last_row_key = ""
    while True:
        if last_row_key:
            params["last_row_key"] = last_row_key
        resp = openapi.get(path, params)
        if not ok(resp):
            print(f"  [WARN] {path}: {resp.get('msg', resp)}")
            break
        data = resp["result"]
        batch = extract_list(data)
        records.extend(batch)
        has_more = data.get("has_more", False) if isinstance(data, dict) else False
        last_row_key = data.get("last_row_key", "") if isinstance(data, dict) else ""
        if not has_more or not last_row_key:
            break
    return records


print("Helpers defined.")

## 1  Token / UID

The `uid` from `connect()` is the cloud project's **service-account** uid — it is NOT an app-user uid.
To query user homes you need the uid of an actual Tuya Smart/Life app user linked to this project.

Two ways to find the real user uid:
- Tuya IoT console → your Cloud project → **Devices** → **Link Devices** → **Users** tab
- Set `TUYA_USER_UID` env var to that value before running

If you don't have it, home IDs will be discovered automatically from the `owner_id` field on each device.


In [ ]:
# The SDK already obtained the token during connect().
# The uid extracted in the cell above is the service-account uid — NOT an app-user uid.
# Do NOT call openapi.get("/v1.0/token") again — the SDK re-signs it incorrectly.

# If TUYA_USER_UID is set (real app user), prefer that over the service-account uid.
APP_USER_UID = os.environ.get("TUYA_APP_USER_UID", "")
print(f"Service-account uid : {USER_UID}")
print(f"App-user uid        : {APP_USER_UID or '(not set — homes will be inferred from device owner_id)'}")

## 2  Homes & Rooms

In [ ]:
all_homes = {}   # home_id -> home dict
all_rooms = {}   # home_id -> [rooms]

# --- Approach A: explicit home IDs from env -----------------------------------
if HOME_IDS_ENV:
    for hid in HOME_IDS_ENV.split(","):
        hid = hid.strip()
        r = openapi.get(f"/v1.0/homes/{hid}")
        if ok(r):
            all_homes[hid] = result(r)
            print(f"Home {hid}: {result(r).get('name', '?')}")
        else:
            print(f"[WARN] home {hid}: {r.get('msg', r)}")

# --- Approach B: real app-user homes -----------------------------------------
if APP_USER_UID:
    print(f"\nFetching homes for app user uid={APP_USER_UID}")
    r = openapi.get(f"/v1.0/users/{APP_USER_UID}/homes")
    pprint(r)
    if ok(r):
        home_list = extract_list(result(r), "homes")
        for h in home_list:
            hid = str(h.get("home_id", h.get("id", "")))
            if hid and hid not in all_homes:
                all_homes[hid] = h
        print(f"Found {len(home_list)} homes via app user UID")
    else:
        print(f"[WARN] user homes: {r.get('msg', r)}")
else:
    print("[INFO] APP_USER_UID not set - will infer homes from device owner_id after devices are fetched")

print(f"\nHomes found so far: {len(all_homes)}")
pprint(list(all_homes.values()))

In [ ]:
# Rooms per home
for hid in all_homes:
    r = openapi.get(f"/v1.0/homes/{hid}/rooms")
    if ok(r):
        rooms = extract_list(result(r), "rooms")
        all_rooms[hid] = rooms
        print(f"Home {hid} rooms ({len(rooms)}):")
        for room in rooms:
            print(f"  {room.get('room_id')} - {room.get('name', '?')}")
    else:
        print(f"[WARN] rooms home {hid}: {r.get('msg', r)}")

## 3  Devices

In [ ]:
all_device_ids  = set()
all_devices_raw = {}   # device_id -> basic info from listing

# --- A: iot-01 associated users/devices (cursor-paged) -----------------------
print("=== iot-01 associated-users devices ===")
assoc = get_cursor_paged("/v1.0/iot-01/associated-users/devices", page_size=50)
print(f"  {len(assoc)} associated devices")
for d in assoc:
    did = d.get("id", d.get("device_id", ""))
    if did:
        all_device_ids.add(did)
        all_devices_raw[did] = d

# --- B: home devices (only if homes known) -----------------------------------
print("\n=== Home devices ===")
for hid in list(all_homes):
    r = openapi.get(f"/v1.0/homes/{hid}/devices")
    if ok(r):
        devs = extract_list(result(r), "devices")
        for d in devs:
            did = d.get("id", d.get("device_id", ""))
            if did:
                all_device_ids.add(did)
                all_devices_raw.setdefault(did, d)
        print(f"  Home {hid}: {len(devs)} devices")
    else:
        print(f"  [WARN] home {hid} devices: {r.get('msg', r)}")

# --- C: user devices (real app user) -----------------------------------------
if APP_USER_UID:
    print("\n=== User devices ===")
    r = openapi.get(f"/v1.0/users/{APP_USER_UID}/devices")
    if ok(r):
        devs = extract_list(result(r), "devices")
        for d in devs:
            did = d.get("id", d.get("device_id", ""))
            if did:
                all_device_ids.add(did)
                all_devices_raw.setdefault(did, d)
        print(f"  {len(devs)} user devices")
    else:
        print(f"  [WARN] {r.get('msg', r)}")

print(f"\nTotal unique device IDs: {len(all_device_ids)}")

# --- D: infer home IDs from device owner_id ----------------------------------
# Each device has an owner_id == home_id. This lets us discover homes even
# without a real app-user uid.
inferred_home_ids = set()
for d in all_devices_raw.values():
    oid = str(d.get("owner_id", ""))
    if oid and oid not in all_homes:
        inferred_home_ids.add(oid)

if inferred_home_ids:
    print(f"\nInferred {len(inferred_home_ids)} home IDs from device owner_id: {inferred_home_ids}")
    for hid in inferred_home_ids:
        r = openapi.get(f"/v1.0/homes/{hid}")
        if ok(r):
            all_homes[hid] = result(r)
            print(f"  Home {hid}: {result(r).get('name', '?')}")
        else:
            # Store a stub so we can still query scenes/automations by home_id
            all_homes[hid] = {"home_id": hid, "name": f"home_{hid}"}
            print(f"  Home {hid}: detail unavailable ({r.get('msg', '')}), storing stub")

print(f"\nTotal homes: {len(all_homes)}")
pprint(list(all_homes.values()))

In [ ]:
# Per-device deep fetch
devices_detail    = {}  # device_id -> full detail
devices_status    = {}  # device_id -> dp status list
devices_functions = {}  # device_id -> functions / instruction set (v1, NO dp_id)
devices_factory   = {}  # device_id -> factory info (local_key, mac, sn)
devices_spec      = {}  # device_id -> iot-03 specification (HAS dp_id when available)
devices_model     = {}  # device_id -> v2 thing model (another dp_id source)
devices_timers    = {}  # device_id -> scheduled tasks
sub_devices       = {}  # gateway_id -> [sub-devices]

device_ids_list = sorted(all_device_ids)
print(f"Fetching details for {len(device_ids_list)} devices...")

for i, did in enumerate(device_ids_list):
    print(f"[{i+1}/{len(device_ids_list)}] {did}")

    # Basic detail
    r = openapi.get(f"/v1.0/devices/{did}")
    if ok(r):
        devices_detail[did] = result(r)
        d = devices_detail[did]
        print(f"    name={d.get('name','?')!r}  ip={d.get('ip','--')}  "
              f"cat={d.get('category','?')}  online={d.get('online')}  "
              f"sub={d.get('sub')}  node_id={d.get('node_id','')}  gw={d.get('gateway_id','')}")
    else:
        print(f"    [WARN] detail: {r.get('msg', r)}")

    # DP status (live values, code-keyed — no dp_id here)
    r = openapi.get(f"/v1.0/devices/{did}/status")
    if ok(r):
        devices_status[did] = result(r)

    # Function / instruction set (v1) — has names but NO dp_id
    r = openapi.get(f"/v1.0/devices/{did}/functions")
    if ok(r):
        devices_functions[did] = result(r)

    # Factory info (local_key, mac, sn)
    r = openapi.get(f"/v1.0/devices/{did}/factory-infos")
    if ok(r):
        devices_factory[did] = result(r)

    # iot-03 product specification — HAS dp_id in each function/status item
    r = openapi.get(f"/v1.0/iot-03/devices/{did}/specification")
    if ok(r):
        spec = result(r)
        devices_spec[did] = spec
        # Count how many DPs actually have dp_id populated
        all_dps = (spec.get("functions") or []) + (spec.get("status") or [])
        with_id = sum(1 for dp in all_dps if dp.get("dp_id"))
        print(f"    spec: {len(all_dps)} DPs, {with_id} have dp_id")
    else:
        print(f"    [WARN] spec: {r.get('msg', r)}")

    # v2 thing model — alternative dp_id source
    r = openapi.get(f"/v2.0/cloud/thing/{did}/model")
    if ok(r):
        devices_model[did] = result(r)

    # Scheduled tasks (timers)
    r = openapi.get(f"/v1.0/devices/{did}/timers", {"type": 0, "device_id": did})
    if ok(r):
        devices_timers[did] = result(r)

    # If gateway, list sub-devices
    if not devices_detail.get(did, {}).get("sub", True):
        r = openapi.get(f"/v1.0/gateways/{did}/sub-devices")
        if ok(r):
            subs = extract_list(result(r))
            if subs:
                sub_devices[did] = subs
                print(f"    gateway has {len(subs)} sub-devices")
                for s in subs:
                    sid = s.get("id", s.get("device_id", ""))
                    if sid:
                        all_device_ids.add(sid)
                        all_devices_raw.setdefault(sid, s)

    time.sleep(0.1)

# ── Product-level functions: best reliable source for dp_ids ──────────────────
# Many devices of the same model share a product_id.
# GET /v1.0/iot-03/products/{product_id}/functions returns dp_id for every DP.
# Query once per unique product_id to avoid redundant API calls.
products_funcs = {}   # product_id -> {category, functions: [{code, dp_id, type, values}]}

unique_product_ids = {
    d.get("product_id", "")
    for d in devices_detail.values()
    if d.get("product_id")
}
print(f"\nFetching product functions for {len(unique_product_ids)} unique product_ids...")

for pid in sorted(unique_product_ids):
    r = openapi.get(f"/v1.0/iot-03/products/{pid}/functions")
    if ok(r):
        products_funcs[pid] = result(r)
        funcs = (result(r) or {}).get("functions") or []
        with_id = sum(1 for f in funcs if f.get("dp_id"))
        print(f"  product {pid}: {len(funcs)} functions, {with_id} have dp_id")
    else:
        print(f"  [WARN] product {pid}: {r.get('msg', r)}")
    time.sleep(0.05)

print("\nDone.")

In [ ]:
# Shadow properties (v2 IoT Core)
devices_shadow = {}  # device_id -> shadow properties

for did in device_ids_list:
    r = openapi.get(f"/v2.0/cloud/thing/{did}/shadow/properties")
    if ok(r):
        devices_shadow[did] = result(r)
    time.sleep(0.05)

print(f"Shadow properties fetched for {len(devices_shadow)} devices.")

## 4  Scenes & Automations

In [ ]:
all_scenes      = {}  # home_id -> [scenes]
all_automations = {}  # home_id -> [automations]

for hid in all_homes:
    # Scenes (tap-to-run)
    r = openapi.get(f"/v1.0/homes/{hid}/scenes")
    if ok(r):
        scenes = extract_list(result(r), "list", "scenes")
        all_scenes[hid] = scenes
        print(f"Home {hid}: {len(scenes)} scenes")
        for s in scenes:
            print(f"  [{s.get('scene_id','?')}] {s.get('name','?')}")
    else:
        print(f"[WARN] scenes home {hid}: {r.get('msg', r)}")

    # Automations
    r = openapi.get(f"/v1.0/homes/{hid}/automations")
    if ok(r):
        autos = extract_list(result(r), "list", "automations")
        all_automations[hid] = autos
        print(f"Home {hid}: {len(autos)} automations")
        for a in autos:
            print(f"  [{a.get('automation_id', a.get('id','?'))}] {a.get('name','?')}")
    else:
        print(f"[WARN] automations home {hid}: {r.get('msg', r)}")

In [ ]:
# Per-scene / automation detail
scenes_detail      = {}  # scene_id -> full detail
automations_detail = {}  # automation_id -> full detail

for hid, scenes in all_scenes.items():
    for s in scenes:
        sid = s.get("scene_id", "")
        if not sid:
            continue
        r = openapi.get(f"/v1.0/homes/{hid}/scenes/{sid}")
        if ok(r):
            scenes_detail[sid] = result(r)
        time.sleep(0.05)

for hid, autos in all_automations.items():
    for a in autos:
        aid = a.get("automation_id", a.get("id", ""))
        if not aid:
            continue
        r = openapi.get(f"/v1.0/homes/{hid}/automations/{aid}")
        if ok(r):
            automations_detail[aid] = result(r)
        time.sleep(0.05)

print(f"Scenes detail   : {len(scenes_detail)}")
print(f"Automations detail: {len(automations_detail)}")

## 5  Cloud Scene Rules (v2 IoT Core)

In [ ]:
cloud_rules        = []
cloud_rules_detail = {}  # rule_id -> full detail

r = openapi.get("/v2.0/cloud/scene/rule", {"page_size": 100, "page_no": 1})
print("Cloud rules:")
pprint(r)

if ok(r):
    cloud_rules = extract_list(result(r), "list", "rules")
    print(f"Found {len(cloud_rules)} cloud rules")
    for rule in cloud_rules:
        rid = rule.get("rule_id", rule.get("id", ""))
        print(f"  [{rid}] {rule.get('name','?')}")
        if rid:
            dr = openapi.get(f"/v2.0/cloud/scene/rule/{rid}")
            if ok(dr):
                cloud_rules_detail[rid] = result(dr)
            time.sleep(0.05)
else:
    print(f"[INFO] Cloud rules not available: {r.get('msg', r)}")

## 6  Device-Scene Bindings (button / scene-switch)

In [ ]:
# Which scenes are bound to which physical buttons/scene-switches
devices_scene_bindings = {}  # device_id -> bound scenes

for did, detail in devices_detail.items():
    r = openapi.get(f"/v1.0/devices/{did}/scenes")
    if ok(r):
        bound = result(r)
        if bound:
            devices_scene_bindings[did] = bound
            name = detail.get("name", did)
            n = len(bound) if isinstance(bound, list) else "?"
            print(f"{name} ({did}): {n} bound scenes")
    time.sleep(0.05)

print(f"\nDevices with scene bindings: {len(devices_scene_bindings)}")

## 7  Device Logs (optional)

In [ ]:
FETCH_LOGS = False   # set True to enable — can be slow for many devices

devices_logs = {}  # device_id -> log entries

if FETCH_LOGS:
    end_ms   = int(time.time() * 1000)
    start_ms = end_ms - 7 * 24 * 3600 * 1000  # last 7 days
    for did in device_ids_list:
        r = openapi.get(f"/v1.0/devices/{did}/logs", {
            "start_row_key": "",
            "start_time": start_ms,
            "end_time":   end_ms,
            "type":       7,
            "size":       100,
        })
        if ok(r):
            devices_logs[did] = result(r)
        time.sleep(0.1)
    print(f"Logs fetched for {len(devices_logs)} devices.")
else:
    print("Logs skipped (set FETCH_LOGS=True to enable).")

## 8  Summary

In [ ]:
print("=" * 60)
print("EXPORT SUMMARY")
print("=" * 60)
print(f"Homes           : {len(all_homes)}")
print(f"Rooms           : {sum(len(v) for v in all_rooms.values())}")
print(f"Devices (unique): {len(all_device_ids)}")
print(f"  with detail   : {len(devices_detail)}")
print(f"  with status   : {len(devices_status)}")
print(f"  with functions: {len(devices_functions)}")
print(f"  with factory  : {len(devices_factory)}")
print(f"  with spec     : {len(devices_spec)}")
print(f"  with model    : {len(devices_model)}")
print(f"  with shadow   : {len(devices_shadow)}")
print(f"  with timers   : {sum(1 for v in devices_timers.values() if v)}")
print(f"  with bindings : {len(devices_scene_bindings)}")
print(f"Gateways (subs) : {len(sub_devices)}")
print(f"Products        : {len(products_funcs)} unique product_ids with functions")
print(f"Scenes          : {sum(len(v) for v in all_scenes.values())} ({len(scenes_detail)} with detail)")
print(f"Automations     : {sum(len(v) for v in all_automations.values())} ({len(automations_detail)} with detail)")
print(f"Cloud rules     : {len(cloud_rules)} ({len(cloud_rules_detail)} with detail)")
print()

# DP-id coverage check per device
print("DP-id coverage per WiFi device:")
print(f"  {'Name':<30} {'product_id':<16} {'spec_dps':>8} {'spec_with_id':>12} {'prod_dps':>8} {'prod_with_id':>12}")
print("  " + "-" * 90)
for did, d in sorted(devices_detail.items(), key=lambda x: x[1].get("name", "").lower()):
    if d.get("sub"):
        continue
    pid = d.get("product_id", "")
    spec = devices_spec.get(did, {})
    spec_dps = (spec.get("functions") or []) + (spec.get("status") or [])
    spec_with_id = sum(1 for dp in spec_dps if dp.get("dp_id"))
    pfuncs = (products_funcs.get(pid) or {}).get("functions") or []
    prod_with_id = sum(1 for f in pfuncs if f.get("dp_id"))
    flag = "" if (spec_with_id or prod_with_id) else "  ← NO dp_ids!"
    print(f"  {d.get('name','?'):<30} {pid:<16} {len(spec_dps):>8} {spec_with_id:>12} {len(pfuncs):>8} {prod_with_id:>12}{flag}")

print()

# Address / network overview table
hdr = f"{'Name':<32} {'Cat':<6} {'IP':<16} {'Online':<7} {'Sub':<5} {'Node ID':<16} {'Gateway ID':<24}"
print("-" * len(hdr))
print(hdr)
print("-" * len(hdr))
for did, d in sorted(devices_detail.items(), key=lambda x: x[1].get("name", "").lower()):
    print(
        f"{d.get('name','?'):<32} "
        f"{d.get('category','?'):<6} "
        f"{d.get('ip','--'):<16} "
        f"{str(d.get('online',False)):<7} "
        f"{str(d.get('sub',False)):<5} "
        f"{d.get('node_id','--'):<16} "
        f"{d.get('gateway_id','--'):<24}"
    )

In [ ]:
# Print scenes / automations in human-readable form
print("\nSCENES (tap-to-run):")
for sid, s in scenes_detail.items():
    print(f"  [{sid}] {s.get('name','?')}")
    for action in s.get("actions", []):
        print(f"    action: {json.dumps(action, ensure_ascii=False)}")

print("\nAUTOMATIONS:")
for aid, a in automations_detail.items():
    print(f"  [{aid}] {a.get('name','?')}  enabled={a.get('enabled')}")
    for cond in a.get("conditions", []):
        print(f"    condition: {json.dumps(cond, ensure_ascii=False)}")
    for action in a.get("actions", []):
        print(f"    action:    {json.dumps(action, ensure_ascii=False)}")

print("\nCLOUD RULES:")
for rid, r in cloud_rules_detail.items():
    print(f"  [{rid}] {r.get('name','?')}")
    pprint(r)

## 9  Save to JSON & CSV

In [ ]:
OUT_DIR = Path("tuya_export")
OUT_DIR.mkdir(exist_ok=True)
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

export = {
    "exported_at": ts,
    "api_endpoint": API_ENDPOINT,
    "homes": all_homes,
    "rooms": all_rooms,
    "devices": {
        "raw":            all_devices_raw,
        "detail":         devices_detail,
        "status":         devices_status,
        "functions":      devices_functions,   # v1, NO dp_id
        "factory":        devices_factory,
        "spec":           devices_spec,        # iot-03, HAS dp_id
        "model":          devices_model,       # v2 thing model, HAS dp_id
        "shadow":         devices_shadow,
        "timers":         devices_timers,
        "sub_devices":    sub_devices,
        "scene_bindings": devices_scene_bindings,
    },
    # Product-level functions: keyed by product_id, each entry has dp_id.
    # Use this as fallback when device-level spec is empty.
    "products_funcs": products_funcs,
    "scenes": {
        "by_home": all_scenes,
        "detail":  scenes_detail,
    },
    "automations": {
        "by_home": all_automations,
        "detail":  automations_detail,
    },
    "cloud_rules": {
        "list":   cloud_rules,
        "detail": cloud_rules_detail,
    },
}
if FETCH_LOGS:
    export["devices"]["logs"] = devices_logs

# Full JSON dump
out_json = OUT_DIR / f"tuya_export_{ts}.json"
with open(out_json, "w", encoding="utf-8") as fh:
    json.dump(export, fh, indent=2, ensure_ascii=False, default=str)
print(f"Saved JSON -> {out_json}  ({out_json.stat().st_size / 1024:.1f} KB)")

# Flat CSV for quick reference
csv_fields = [
    "device_id", "name", "category", "product_id", "model",
    "ip", "online", "sub", "node_id", "gateway_id",
    "local_key", "mac", "sn",
    "owner_id", "room_id",
    "active_time", "update_time", "create_time",
]
out_csv = OUT_DIR / f"devices_{ts}.csv"
with open(out_csv, "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=csv_fields, extrasaction="ignore")
    writer.writeheader()
    for did in sorted(all_device_ids):
        row = {"device_id": did}
        row.update(devices_detail.get(did, all_devices_raw.get(did, {})))
        fi = devices_factory.get(did)
        if fi:
            fi_item = fi[0] if isinstance(fi, list) and fi else fi
            if isinstance(fi_item, dict):
                for k in ("local_key", "mac", "sn"):
                    if k in fi_item:
                        row[k] = fi_item[k]
        writer.writerow(row)
print(f"Saved CSV  -> {out_csv}")